#**Fonctionnement du code microGPT de Andrej Karpathy**

#**Partie 1 : explication du code**

L'objectif de ce code est de créer, à partir d'une base de prénoms, de nouveaux prénoms innexistants.
Pour cela A.Karpathy, en se basant sur une architecture GPT (Generative Pre-trained Transformer), a écrit un code en python pur de seulement **~200 lignes** à l'aide des bibliothèques de bases de python.
Ce code se décompose en **6 parties** :

## 0-Importation des bibliothèques et du dataset (depuis github)

In [ ]:
import os       # os.path.exists
import math     # math.log, math.exp
import random   # random.seed, random.choices, random.gauss, random.shuffle
random.seed(42) # Let there be order among chaos
import time     #Ajout personnel pour mesurer le temps


# Let there be a Dataset `docs`: list[str] of documents (e.g. a list of names)
if not os.path.exists('input.txt'):
    import urllib.request
    names_url = 'https://raw.githubusercontent.com/karpathy/makemore/988aa59/names.txt'
    urllib.request.urlretrieve(names_url, 'input.txt')
docs = [line.strip() for line in open('input.txt') if line.strip()]
random.shuffle(docs)
print(f"Nombre de prénoms: {len(docs)}") # Affiche le nombre de prénoms présents dans le dataset

l_max_nom=max(len(name) for name in docs)
print(f"Longeur du prénom le plus long: {l_max_nom}")



Nombre de prénoms: 32033
Longeur du prénom le plus long: 15


## 1-Le Tokenizer

Le Tokenizer **transforme chaque caractère en un entier unique (*son index dans uchars*)**. Ces entiers sont appelés tokens. Pour les prénoms, il y en aura 27 : 26, 1 pour chaque lettre (ici il n'y a pas de caractère spéciaux tels que des tirets ou des accents), +1 pour le BOS, qui marque le début ET la fin d'un prénom.


In [ ]:
# Let there be a Tokenizer to translate strings to sequences of integers ("tokens") and back
uchars = sorted(set(''.join(docs))) # L'id de chaque token est sa position dans la liste
BOS = len(uchars) # L'Id du BOS est égal à l'Id du dernier token du uchar+1
vocab_size = len(uchars) + 1 # Nombre de tokens
print(f"Nombre de Tokens différents crées (BOS compris): {vocab_size}")
print(f"Liste des tokens: {uchars}")




Nombre de Tokens différents crées (BOS compris): 27
Liste des caractères: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Dictionnaire des tokens: {'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5, 'g': 6, 'h': 7, 'i': 8, 'j': 9, 'k': 10, 'l': 11, 'm': 12, 'n': 13, 'o': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'v': 21, 'w': 22, 'x': 23, 'y': 24, 'z': 25, 'BOS': 26}


## 2-L'autograd de microGPT : une reprise simplifiée du mécanisme de l'autograd PyTorch


*Autograd* : c'est un mécanisme permettant à un réseau de neurones de comprendre ses erreurs et de les corriger

Les étapes de **l'Autograd** :

  i-*Forward pass* (l'Aller) :
  Cette étape commence dès le départ et permet d'enregistrer chaque opération dans un graphe de calcul.

  ii- *Backward pass* (le Retour) :
  Cette étape s'exécute après le transformer et permet de remonter le graphe pour calculer le gradient de chaque poids (avec    la règle de la chaîne).

In [ ]:
# Let there be Autograd to recursively apply the chain rule through a computation graph
class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads') # Python optimization for memory usage

#Initialisation de tout les calculs de base pour le Forward Pass

    def __init__(self, data, children=(), local_grads=()):
        self.data = data                # scalar value of this node calculated during forward pass
        self.grad = 0                   # derivative of the loss w.r.t. this node, calculated in backward pass
        self._children = children       # children of this node in the computation graph
        self._local_grads = local_grads # local derivative of this node w.r.t. its children

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, other): return Value(self.data**other, (self,), (other * self.data**(other-1),))
    def log(self): return Value(math.log(self.data), (self,), (1/self.data,))
    def exp(self): return Value(math.exp(self.data), (self,), (math.exp(self.data),))
    def relu(self): return Value(max(0, self.data), (self,), (float(self.data > 0),))
    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return other * self**-1

# Definition du Backward Pass

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad


## 3-Création d'un vecteur pour identifier un token et sa position : WTE + WPE

Lorsque l'on veut analyser un token, il n'est pas directement exploitable car il est sous la forme d'un identifiant.
L'objectif est donc de créer un vecteur de dimension 16 pour être exploité par les matrices du transformer.

(NB: La dimension 16 est un choix arbitraire de A.Karpathy car il est adapté au modèle microGPT et un dataset d'environ 32 000 prénoms.
A titre de comparaison, pour le modèle GPT-2-small la dimension est de 768 car le dataset correspond à internet tout entier,
pour le modèle GPT-3 la dimension est de 12 288 car le dataset correspond à internet x 10,)


  **i- WTE ( Word Token Embedding)** : Transforme l'id du token en vecteur de dimension 16

  **ii- WPE (Word Position Embedding)** : Transforme l'id de la position du token en vecteur de dimension 16

**WTE** et **WPE** sont des **matrices de poids**. Un poids c'est ce qui va permettre de donner plus ou moins d'**importance à un paramètre**. L'utilisation d'une matrice de poids permet donc d'accocrder un intérêt différents à chacune des composante de notre vecteur. *Chaque poids est initialisé aléatoirement et sera ajustés pendant l'entraînement*.

In [ ]:
# Initialize the parameters, to store the knowledge of the model
n_layer = 1     # depth of the transformer neural network (number of layers)
n_embd = 16   # width of the network (embedding dimension)
block_size = 16 # maximum context length of the attention window (note: the longest name is 15 characters)
n_head = 4      # number of attention heads
head_dim = n_embd // n_head # derived dimension of each head
matrix = lambda nout, nin, std=0.08: [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]    # Initialisation de la matrice avec les poids non entraînés
state_dict = {'wte': matrix(vocab_size, n_embd), 'wpe': matrix(block_size, n_embd), 'lm_head': matrix(vocab_size, n_embd)}



# Au départ chaque vecteur représentant l'id du token correspond à une ligne de la matrice (de même pour la position avec wpe)
#Le vecteur de la lettre 'a' est donc la première ligne de la matrice

tok_emb_a= state_dict['wte'][0]
pos_emb_a = state_dict['wpe'][0]

print('Vecteur identité de la lettre '+'a'+f': {[v.data for v in (tok_emb_a)]}')
print('Vecteur position de la lettre '+'a'+f': {[v.data for v in (pos_emb_a)]}')


Vecteur identité de la lettre a: [-0.04273180935726127, 0.07696138795865093, 0.10844210106107166, 0.03741680212434131, -0.0031437655498982994, -0.004425542236071316, 0.09739277609170183, -0.11227107651933786, -0.03318121711190419, -0.04985948901569463, 0.01114614081126478, -0.028872684050893608, 0.046982695064467506, -0.017899943290165567, -0.0005483556527782804, -0.012542092727380863]
Vecteur position de la lettre a: [-0.02223609248240166, -0.06955846384028579, -0.18050460245283062, 0.05687695766770791, 0.047400660094667646, -0.027228818329571906, -0.06230811240334903, 0.07152032540868632, 0.11822576631407689, -0.10224950084969167, 0.04580701264968371, 0.10446673348803803, 0.03989979579720061, 0.024080426158084624, -0.11147360119730007, 0.07657175167226077]


Enfin on **somme** les deux vecteurs pour obtenir un vecteur X contenant les deux informations.

In [ ]:
x = [t + p for t, p in zip(tok_emb_a, pos_emb_a)]
print('Vecteur initial WTE+WPE de '+'a'+f': {[v.data for v in (x)]}')


Vecteur initial WTE+WPE de a: [-0.06496790183966293, 0.007402924118365142, -0.07206250139175896, 0.09429375979204921, 0.044256894544769346, -0.03165436056564322, 0.0350846636883528, -0.040750751110651545, 0.08504454920217269, -0.1521089898653863, 0.056953153460948494, 0.07559404943714443, 0.08688249086166812, 0.006180482867919057, -0.11202195685007835, 0.0640296589448799]


## 4- Le GPT: le coeur du modèle

Il **prend en entrée notre vecteur X afin d'en resortir un nombre de probabilité** égale au nombre de token. Chaque probabilité associé au token T correspond au **pourcentage de chance que le token suivant soit le token T**.
Le GPT est constitué de 2 parties :

i-Le **Transformer** est constitué d'un enchaînement de 4 blocs et de 2 connexions residuelles:
    
1) **RMSNorm (Root Mean Square Norm)** : Il permet de *mettre chaque valeur à une échelle proche de 1* pour éviter les valeurs trop basses ou trop élévée


In [ ]:
x_bis=x
def rmsnorm(x):
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]
x_norme=rmsnorm(x)
print(f'Normalisation de notre vecteur x représentant la lettre a {[v.data for v in (x_norme)]}')


Normalisation de notre vecteur x représentant la lettre a [-0.8763482244391056, 0.09985761003637766, -0.9720468624516102, 1.2719230055079547, 0.5969786595419868, -0.42698381649938477, 0.4732549744981985, -0.5496844960225774, 1.1471609453462208, -2.0517892592365854, 0.7682377527734713, 1.0196837072851235, 1.171952832790832, 0.08336817157562185, -1.511057617745434, 0.863692320959801]


2) **L'attention** : Se base sur **trois variables** : *q (la requête), k (la clé), v (la valeur)*.   D'abord on effectue le *produit scalaire* entre Q (du token actuel) et K (des tokens passés). Le résultat obtenu est un **score** *qui plus il est élevé, plus ce token est pertinent*. Ensuite on le normalise en pourcentage avec **softmax()**. Enfin on multiplie les poids de softmax() par les Valeurs et on **ajoute le résultat à notre token** pour qu'il enregistre ce que les tokens précédent lui ont "appris".

Pour ce faire on effectue au préalable quelques étapes:

  - On crée une fonction **linear()** pour effectuer les produits matriciels ainsi que la fonction **softmax()** ( # Bloc  1 #)
  - On crée des matrices permettant de **stocker les paramètres** (k,v,...) de chaque lettre pour ne pas avoir à les recalculer quand l'attention va regarder les lettres précédentes (# Bloc 2 #)



In [ ]:
# Bloc 1 #

def linear(x, w): # Fonction permettant d'effectuer un produit matriciel
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]

def softmax(logits):
    max_val = max(val.data for val in logits)
    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)
    return [e / total for e in exps]

In [ ]:
# Bloc 2 #

for i in range(n_layer):
    state_dict[f'layer{i}.attn_wq'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wk'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wv'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wo'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc1'] = matrix(4 * n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc2'] = matrix(n_embd, 4 * n_embd)
params = [p for mat in state_dict.values() for row in mat for p in row] # flatten params into a single list[Value]
print(f"Nombre de paramètres (somme du nombre des dimensions de chaque matrice de state_dict): {len(params)}")

Nombre de paramètres (somme du nombre des dimensions de chaque matrice de state_dict): 4192


In [ ]:
#Il s'agit de la première partie de la fonction GPT

def attention(token_id, pos_id, keys, values):
    tok_emb = state_dict['wte'][token_id] # token embedding
    pos_emb = state_dict['wpe'][pos_id] # position embedding
    x = [t + p for t, p in zip(tok_emb, pos_emb)] # joint token and position embedding
    x = rmsnorm(x) # note: not redundant due to backward pass via the residual connection

    for li in range(n_layer): #Ici la boucle n'est pas nécessaire car Karpanthy a choisi n_layer=1
        # Attention
        x_residual = x
        x = rmsnorm(x)
        q = linear(x, state_dict[f'layer{li}.attn_wq'])
        k = linear(x, state_dict[f'layer{li}.attn_wk'])
        v = linear(x, state_dict[f'layer{li}.attn_wv'])
        keys[li].append(k)
        values[li].append(v)
        x_attn = []
        for h in range(n_head):
            hs = h * head_dim
            q_h = q[hs:hs+head_dim]
            k_h = [ki[hs:hs+head_dim] for ki in keys[li]]
            v_h = [vi[hs:hs+head_dim] for vi in values[li]]
            attn_logits = [sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim**0.5 for t in range(len(k_h))]
            attn_weights = softmax(attn_logits)
            head_out = [sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h))) for j in range(head_dim)]
            x_attn.extend(head_out)
        x = linear(x_attn, state_dict[f'layer{li}.attn_wo'])
        return(x)
x_attention=attention(0,0,[[]],[[]])
print(f"Vecteur x après l'attention :{[v.data for v in (x_attention)]}")

Vecteur x après l'attention :[0.05605006229878849, 0.00567921456921205, -0.1234656717375192, -0.06147900095604581, -0.0961211617439424, -0.17750653154890067, -0.12735040386306673, 0.3157388112929311, -0.10275473854399168, 0.046516022979404166, -0.06521405378632379, -0.0013998881999841132, 0.03093982439196155, 0.015981061141883822, -0.2480392144145799, -0.011953256344230877]


On effectue ensuite une **connexion résiduelle** pour **additioner** le token original et le résultat de l'attention.

In [ ]:
x_residual = x_attention
x = [a + b for a, b in zip(x_residual,x)]
print (f'Vecteur x après connexion residuelle: {[v.data for v in (x)]}')

Vecteur x après connexion residuelle: [-0.00891783954087444, 0.013082138687577193, -0.19552817312927817, 0.0328147588360034, -0.05186426719917306, -0.2091608921145439, -0.09226574017471392, 0.2749880601822795, -0.01771018934181899, -0.10559296688598213, -0.008260900325375298, 0.07419416123716031, 0.11782231525362967, 0.02216154400980288, -0.36006117126465825, 0.052076402600649024]


3) Une autre utilisation de **RMSNorm** pour remettre à l'échelle

In [ ]:
x_norme2=rmsnorm(x)
print(f'Normalisation de notre vecteur x : {[v.data for v in (x_norme2)]}')

Normalisation de notre vecteur x : [-0.06171791456879961, 0.09053788355311758, -1.353196705286644, 0.2271019200506531, -0.3589383277755286, -1.4475450036403708, -0.638545809624613, 1.9031167277658478, -0.12256734916408601, -0.7307798799770857, -0.05717141895254247, 0.513477382470351, 0.8154158362899474, 0.15337395045522323, -2.491884329808776, 0.3604064585403066]


4) Le **MLP (Multi-Layer Perceptron)**: Après avoir obtenu de nombreuses informations d'apprentissage via l'attention, le MLP va permettre de les **transformer en des données exploitables**. Pour ce faire on **agrandit les dimensions** avec la fonction *.mlp_fc1* de notre vecteur (ici on passe de 16 à 64) puis on utilise la fonction *.relu()* (relu(x) = max(0,x) )pour **mettre à 0 toutes les valeurs négatives** de notre vecteur, enfin **on repasse en dimension 16** avec la fonction *.mlp_fc2*. La fonction *.relu()* (comme n'importe quelle autre *fonction non linéaire*) permet d'empécher l'obtention d'une fonction linéaire, qui ne serait pas assez complexe pour notre modèle, (ici Karpathy utilise *relu* car c'est la fonction non-linéaire la plus simple possible).

In [ ]:
#Il s'agit de la deuxième partie de la fonction GPT

li=0 # on prend li=0 car il n'y a qu'une seule boucle

def mlp(x):
  x_mlp= x
  x_mlp= rmsnorm(x)
  x_mlp= linear(x, state_dict[f'layer{li}.mlp_fc1'])
  x_mlp= [xi.relu() for xi in x]
  x_mlp= linear(x, state_dict[f'layer{li}.mlp_fc2'])
  return(x_mlp)

x_mlp=mlp(x_norme2)
print(f'Vecteur x après le MLP: {[v.data for v in (x_mlp)]}')

Vecteur x après le MLP: [-0.485562268497372, 0.12171515077998415, 0.03822089872676047, -0.3504677778639684, 0.18878921424102005, 0.2786724396594238, 0.2472701933030565, -0.10587465591131034, 0.411674777508321, 0.04448688797170075, 0.48256925773909054, 0.44618964990089927, 0.049463403218154664, 0.09915338075466344, -0.3247827029766284, 0.5930641612239025]


On effectue ensuite une **connexion résiduelle** pour additioner le token avant application du MLP et le résultat du MLP.

In [ ]:
x_residual=x_mlp
x = [a + b for a, b in zip(x_residual,x)]
print(f'Vecteur x après connexion residuelle: {x}')

Vecteur x après connexion residuelle: [<__main__.Value object at 0x78988169f440>, <__main__.Value object at 0x78988169c500>, <__main__.Value object at 0x78988169c3c0>, <__main__.Value object at 0x78988169c300>, <__main__.Value object at 0x78988169c2c0>, <__main__.Value object at 0x78988169c4c0>, <__main__.Value object at 0x78988169c480>, <__main__.Value object at 0x78988169c6c0>, <__main__.Value object at 0x78988169c880>, <__main__.Value object at 0x78988169c1c0>, <__main__.Value object at 0x78988169c840>, <__main__.Value object at 0x78988169c640>, <__main__.Value object at 0x78988169c8c0>, <__main__.Value object at 0x78988169c680>, <__main__.Value object at 0x78988169f840>, <__main__.Value object at 0x78988169fac0>]


ii-**Génération des probabilités**:

Pour terminer on va générer nos **probabilités d'obtenir chaque caractère** après le passage dans le transformer. Pour ce faire on effectue un **produit matriciel entre la de poids lm_head** crée au départ et notre vecteur X après passage dans le transformer

In [ ]:
logits = linear(x, state_dict['lm_head'])
print(f"Les probabilités des caractères venant juste après a sont (sous forme de vecteur) :{[v.data for v in (softmax(logits))]}")

print("Les probabilités cachées derrière chaque éléments du vecteur softmax(logits) sont:")
logits = linear(x, state_dict['lm_head'])
probs = softmax(logits)
for i, p in enumerate(probs):
    label = uchars[i] if i < len(uchars) else 'BOS'
    print(f"{label}: {p.data*100:.2f}%")
print("Ces probabilités ne sont pas utilisées tel quel par le script, il utilise celle dans le vecteur softmax(logits)")

Les probabilités des caractères venant juste après a sont (sous forme de vecteur) :[0.03410331832611886, 0.03486763659753766, 0.034232018867398815, 0.034038128550981026, 0.04346281869734156, 0.03353761012661196, 0.03487430553928042, 0.034842121520384454, 0.03755628211587888, 0.03245692621698513, 0.040062389411991514, 0.03997603108944041, 0.03625849950930387, 0.037359931906578644, 0.04515571806651686, 0.03425543446339692, 0.03449861850544969, 0.035427048026512045, 0.03728315371916851, 0.0416217287962403, 0.037256761562714846, 0.034625007642148695, 0.033645666333733025, 0.03506268994828947, 0.03714451499184211, 0.04387722561179541, 0.04251841385635882]
Les probabilités cachées derrière chaque éléments du vecteur softmax(logits) sont:
a: 3.41%
b: 3.49%
c: 3.42%
d: 3.40%
e: 4.35%
f: 3.35%
g: 3.49%
h: 3.48%
i: 3.76%
j: 3.25%
k: 4.01%
l: 4.00%
m: 3.63%
n: 3.74%
o: 4.52%
p: 3.43%
q: 3.45%
r: 3.54%
s: 3.73%
t: 4.16%
u: 3.73%
v: 3.46%
w: 3.36%
x: 3.51%
y: 3.71%
z: 4.39%
BOS: 4.25%
Ces probabili

**Code GPT entier**

In [ ]:

def gpt(token_id, pos_id, keys, values): #(WTE,WPE,)
    tok_emb = state_dict['wte'][token_id] # token embedding
    pos_emb = state_dict['wpe'][pos_id] # position embedding
    x = [t + p for t, p in zip(tok_emb, pos_emb)] # joint token and position embedding
    x = rmsnorm(x) # note: not redundant due to backward pass via the residual connection

    for li in range(n_layer):
        # 1) Multi-head Attention block
        x_residual = x
        x = rmsnorm(x)
        q = linear(x, state_dict[f'layer{li}.attn_wq'])
        k = linear(x, state_dict[f'layer{li}.attn_wk'])
        v = linear(x, state_dict[f'layer{li}.attn_wv'])
        keys[li].append(k)
        values[li].append(v)
        x_attn = []
        for h in range(n_head):
            hs = h * head_dim
            q_h = q[hs:hs+head_dim]
            k_h = [ki[hs:hs+head_dim] for ki in keys[li]]
            v_h = [vi[hs:hs+head_dim] for vi in values[li]]
            attn_logits = [sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim**0.5 for t in range(len(k_h))]
            attn_weights = softmax(attn_logits)
            head_out = [sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h))) for j in range(head_dim)]
            x_attn.extend(head_out)
        x = linear(x_attn, state_dict[f'layer{li}.attn_wo'])
        x = [a + b for a, b in zip(x, x_residual)]
        # 2) MLP block
        x_residual = x
        x = rmsnorm(x)
        x = linear(x, state_dict[f'layer{li}.mlp_fc1'])
        x = [xi.relu() for xi in x]
        x = linear(x, state_dict[f'layer{li}.mlp_fc2'])
        x = [a + b for a, b in zip(x, x_residual)]

    logits = linear(x, state_dict['lm_head'])
    return logits
v_proba=[v.data for v in (gpt(0,0,[[]],[[]]))]
print(f"Le vecteur associé au probabilité est : {v_proba}")

Le vecteur associé au probabilité est : [-0.17855938161296783, 0.20715299164505777, 0.32624210326869396, -0.07933367154066077, 0.16817058727810785, -0.16488108253749675, -0.06873092165383457, -0.016510472763677966, 0.1335530065511597, -0.3082203326007231, 0.2599154230106877, 0.5153991573136691, -0.31129865053945094, 0.014994053146464664, 0.6482866871067574, 0.3533141456852139, -0.044823705739803515, -0.12508930214715724, 0.3655951203590097, 0.297012069677526, -0.3039020219196231, -0.18332609124955807, -0.21352596603023088, -0.4733108244678408, 0.2623894477863638, 0.19334883342251472, 0.49828942309382135]


## 5-Boucle d'entraînement : la phase d'apprentissage

On crée une boucle qui répète un nombre arbitraire de fois les étapes précédentes afin **d'entraîner notre modèle** : ( *Exemple* de boucle avec le prénom '*penelope*')

1) On prend un prénom de notre dataset que l'on convertit en token

In [ ]:
step,num_steps=0,1
doc = docs[0]
tokens = [26, 15, 4, 13, 4, 11, 14, 15, 4, 26] #Token correspondant au prénom Penelope
n = min(block_size, len(tokens) - 1)
print(tokens)
print (n)

[26, 15, 4, 13, 4, 11, 14, 15, 4, 26]
9


2) **Prédiction du prochain token** et mesure de la **pertinance des probabilités** ( placée dans une variable de perte notée **loss**). Plus la **perte est élevée** et **moins les probabilités seront pertinantes.**
On applique ensuite le *backward()* pass de notre autograd afin de **calculer le gradient** des calculs enregistrés depuis le début du proccesus.

In [ ]:
keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
losses = []
for pos_id in range(n):
    token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
    logits = gpt(token_id, pos_id, keys, values)
    probs = softmax(logits)
    loss_t = -probs[target_id].log()
    losses.append(loss_t)
loss = (1 / n) * sum(losses)

loss.backward() # Calcul du gradient

print (f"La valeur de la perte est de :{loss.data}")

La valeur de la perte est de :3.343907412802732


3) Mise à jour des poids avec l'optimisation **Adam** (Adaptative Moment Estimation) :

Adam est une méthode d'optimisation se basant sur l'ajout de 2 variables pour tracer une moyenne pondérée des gradients du *backwardpass* où les gradients les plus récents comptent plus que les anciens.
Le premier est le momentum (m): il permet l'accélaration (resp. décélération) de la vitesse d'apprentissage lorsque les gradients sont dans la même direction (resp. dans des directions opposées), car cela signifie que les paramètres sont cohérents (resp. incohérent) entre eux.
Le deuxième est l'adaptation du taux d'apprentissage (v): il permet d'adapter la vitesse d'apprentissage en fonction de la valeur du gradient: plus un gradient est grand, plus le taux d'apprentissage diminue et inversemment.

In [ ]:
# Paramètres de base de de l'optimisation Adam
learning_rate, beta1, beta2, eps_adam = 0.01, 0.85, 0.99, 1e-8
m = [0.0] * len(params) # momentum
v = [0.0] * len(params) # taux d'apprentissage

In [ ]:
wte_e_avant = [v.data for v in state_dict['wte'][4]]
lr_t = learning_rate * (1 - step / num_steps) # linear learning rate decay
for i, p in enumerate(params):
  m[i] = beta1 * m[i] + (1 - beta1) * p.grad
  v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2
  m_hat = m[i] / (1 - beta1 ** (step + 1))
  v_hat = v[i] / (1 - beta2 ** (step + 1))
  p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
  p.grad = 0
wte_e_après = [v.data for v in state_dict['wte'][4]]
print(f"step {step+1:4d} / {num_steps:4d} | loss {loss.data:.4f}", end='\n')
#On remarque l'ajustement des vecteurs correspondant à 'e' dans WTE (on a également un changement de tout les autres vecteurs de chaque lettre du prénom penelope ainsi que du BOS)
# NB: si l'on avait comparé les vecteurs WTE avant et après d'une lettre qui n'appartient pas au prénom penelope, les deux vecteurs resterait inchangés
print(f"Vecteur dans WTE correspondant à la lettre e avant : {wte_e_avant}")
print(f"Vecteur dans WTE correspondant à la lettre e après : {wte_e_après}")


step    1 /    1 | loss 3.3439
Vecteur dans WTE correspondant à la lettre e avant : [-0.019867990465001373, 0.053340695870774966, 0.046320835075806234, 0.03411367875611231, 0.023184555170819018, 0.04501820856920581, -0.1555436081323037, 0.14822741498746517, 0.013559067625421058, -0.044970649996024195, -0.00655564287596722, -0.06091469552763412, 0.040786496453622804, -0.07160684377033148, -0.00863471509045191, 0.0406546670434344]
Vecteur dans WTE correspondant à la lettre e après : [-0.029867989715283462, 0.04334069663101738, 0.056320832725392844, 0.04411367800985817, 0.03318455427766263, 0.03501822434528179, -0.14554360855794732, 0.13822741612508221, 0.02355906601682476, -0.034970651461173126, 0.003444355067840324, -0.07091469116665274, 0.030786520277961055, -0.08160684329331652, -0.018634714616496125, 0.050654666637373134]


**Boucle Complète** :

In [ ]:
# Paramètres de base de de l'optimisation Adam
learning_rate, beta1, beta2, eps_adam = 0.01, 0.85, 0.99, 1e-8
m = [0.0] * len(params) # momentum
v = [0.0] * len(params) # taux d'apprentissage

num_steps = 1000 # nombre de boucle
for step in range(num_steps):

    # Choix d'un nom du data set
    doc = docs[step % len(docs)]
    tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
    n = min(block_size, len(tokens) - 1)

    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    losses = []
    for pos_id in range(n):
        token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax(logits)
        loss_t = -probs[target_id].log()
        losses.append(loss_t)
    loss = (1 / n) * sum(losses) # perte moyenne

    # Backward the loss, calculating the gradients with respect to all model parameters
    loss.backward()

    # Optimisation Adam
    lr_t = learning_rate * (1 - step / num_steps) # linear learning rate decay
    for i, p in enumerate(params):
        m[i] = beta1 * m[i] + (1 - beta1) * p.grad
        v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2
        m_hat = m[i] / (1 - beta1 ** (step + 1))
        v_hat = v[i] / (1 - beta2 ** (step + 1))
        p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
        p.grad = 0

    print(f"step {step+1:4d} / {num_steps:4d} | loss {loss.data:.4f}", end='\n')

step    1 / 1000 | loss 3.3319
step    2 / 1000 | loss 3.3966
step    3 / 1000 | loss 3.2047
step    4 / 1000 | loss 3.1015
step    5 / 1000 | loss 3.1845
step    6 / 1000 | loss 2.9233
step    7 / 1000 | loss 3.1732
step    8 / 1000 | loss 3.3257
step    9 / 1000 | loss 2.7965
step   10 / 1000 | loss 3.1343
step   11 / 1000 | loss 2.7289
step   12 / 1000 | loss 2.8604
step   13 / 1000 | loss 3.0452
step   14 / 1000 | loss 3.0193
step   15 / 1000 | loss 3.0855
step   16 / 1000 | loss 2.7412
step   17 / 1000 | loss 2.8753
step   18 / 1000 | loss 2.8027
step   19 / 1000 | loss 2.7249
step   20 / 1000 | loss 2.6961
step   21 / 1000 | loss 3.7184
step   22 / 1000 | loss 2.7155
step   23 / 1000 | loss 2.7895
step   24 / 1000 | loss 2.0034
step   25 / 1000 | loss 3.4437
step   26 / 1000 | loss 2.8762
step   27 / 1000 | loss 3.2914
step   28 / 1000 | loss 2.9762
step   29 / 1000 | loss 2.2795
step   30 / 1000 | loss 2.3473
step   31 / 1000 | loss 2.8819
step   32 / 1000 | loss 2.9800
step   3

## 6-L'Inférence : la génération des nouveaux prénoms

C'est l'étape de l'obtention des résultats. Pour former un nouveau prénom le script procède de la manière suivante: On prend notre **token de départ** (le BOS), on calcule à l'aide du transformer les différentes **probabilités** pour savoir quelle lettre a le plus de chance de venir après, puis on l'**ajoute à notre prénom**.

On répète la boucle **probabilités** -> **ajout de lettre** jusqu'à retomber sur le token BOS, symbolisant le début d'un nouveau prénom.
Le paramètre *temperature* sert ici à gérer le taux de créativité des prénoms générés. Plus ce paramètre est proche de 0, plus les prénoms vont se ressembler. Plus le paramètre sera grand, plus l'algorithme prend des libertés sur le choix de la lettre suivant en prenant de moins en moins en compte les probabilités (*probabilité d'un token*= softmax(valeur en sortie du gpt divisé par la *température*)

In [ ]:
temperature = 0.5 # in (0, 1], control the "creativity" of generated text, low to high
print("\n--- inference (new, hallucinated names) ---")
for sample_idx in range(20):
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    token_id = BOS
    sample = []
    for pos_id in range(block_size):
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax([l / temperature for l in logits])
        token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
        if token_id == BOS:
            break
        sample.append(uchars[token_id])
    print(f"sample {sample_idx+1:2d}: {''.join(sample)}")


--- inference (new, hallucinated names) ---
sample  1: jamny
sample  2: ann
sample  3: karaly
sample  4: iarer
sample  5: nalah
sample  6: karia
sample  7: winan
sample  8: anna
sample  9: asan
sample 10: sanare
sample 11: konla
sample 12: keylen
sample 13: keran
sample 14: alerin
sample 15: daran
sample 16: lemie
sample 17: kana
sample 18: lara
sample 19: alela
sample 20: anyma


##**Code microGPT Complet**

In [ ]:
import os       # os.path.exists
import math     # math.log, math.exp
import random   # random.seed, random.choices, random.gauss, random.shuffle
random.seed(42) # Let there be order among chaos

def code_karpathy(n_layer=1,n_embd=16,num_steps=1000,temperature=0.5):
  t0 = time.time()
  # Let there be a Dataset `docs`: list[str] of documents (e.g. a list of names)
  if not os.path.exists('input.txt'):
      import urllib.request
      names_url = 'https://raw.githubusercontent.com/karpathy/makemore/988aa59/names.txt'
      urllib.request.urlretrieve(names_url, 'input.txt')
  docs = [line.strip() for line in open('input.txt') if line.strip()]
  random.shuffle(docs)
  print(f"num docs: {len(docs)}")

  ### Tokenizer ###

  # Let there be a Tokenizer to translate strings to sequences of integers ("tokens") and back
  uchars = sorted(set(''.join(docs))) # unique characters in the dataset become token ids 0..n-1
  BOS = len(uchars) # token id for a special Beginning of Sequence (BOS) token
  vocab_size = len(uchars) + 1 # total number of unique tokens (nb lettres alphabet  +1 ( for BOS)
  print(f"vocab size: {vocab_size}")

  ### Autograd ###

  # Let there be Autograd to recursively apply the chain rule through a computation graph
  class Value:
      __slots__ = ('data', 'grad', '_children', '_local_grads') # Python optimization for memory usage

      def __init__(self, data, children=(), local_grads=()):
          self.data = data                # scalar value of this node calculated during forward pass
          self.grad = 0                   # derivative of the loss w.r.t. this node, calculated in backward pass
          self._children = children       # children of this node in the computation graph
          self._local_grads = local_grads # local derivative of this node w.r.t. its children

      def __add__(self, other):
          other = other if isinstance(other, Value) else Value(other)
          return Value(self.data + other.data, (self, other), (1, 1))

      def __mul__(self, other):
          other = other if isinstance(other, Value) else Value(other)
          return Value(self.data * other.data, (self, other), (other.data, self.data))

      def __pow__(self, other): return Value(self.data**other, (self,), (other * self.data**(other-1),))
      def log(self): return Value(math.log(self.data), (self,), (1/self.data,))
      def exp(self): return Value(math.exp(self.data), (self,), (math.exp(self.data),))
      def relu(self): return Value(max(0, self.data), (self,), (float(self.data > 0),))
      def __neg__(self): return self * -1
      def __radd__(self, other): return self + other
      def __sub__(self, other): return self + (-other)
      def __rsub__(self, other): return other + (-self)
      def __rmul__(self, other): return self * other
      def __truediv__(self, other): return self * other**-1
      def __rtruediv__(self, other): return other * self**-1

      def backward(self):
          topo = []
          visited = set()
          def build_topo(v):
              if v not in visited:
                  visited.add(v)
                  for child in v._children:
                      build_topo(child)
                  topo.append(v)
          build_topo(self)
          self.grad = 1
          for v in reversed(topo):
              for child, local_grad in zip(v._children, v._local_grads):
                  child.grad += local_grad * v.grad

  ### WTE / WPE ###

  # Initialize the parameters, to store the knowledge of the model
  block_size = 16 # maximum context length of the attention window (note: the longest name is 15 characters)
  n_head = 4      # number of attention heads
  head_dim = n_embd // n_head # derived dimension of each head
  matrix = lambda nout, nin, std=0.08: [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]    # Initialisation de la matrice avec les poids non entraînés
  state_dict = {'wte': matrix(vocab_size, n_embd), 'wpe': matrix(block_size, n_embd), 'lm_head': matrix(vocab_size, n_embd)}

  for i in range(n_layer):
      state_dict[f'layer{i}.attn_wq'] = matrix(n_embd, n_embd)
      state_dict[f'layer{i}.attn_wk'] = matrix(n_embd, n_embd)
      state_dict[f'layer{i}.attn_wv'] = matrix(n_embd, n_embd)
      state_dict[f'layer{i}.attn_wo'] = matrix(n_embd, n_embd)
      state_dict[f'layer{i}.mlp_fc1'] = matrix(4 * n_embd, n_embd)
      state_dict[f'layer{i}.mlp_fc2'] = matrix(n_embd, 4 * n_embd)
  params = [p for mat in state_dict.values() for row in mat for p in row] # flatten params into a single list[Value]
  print(f"num params: {len(params)}")

  # Define the model architecture: a function mapping tokens and parameters to logits over what comes next
  # Follow GPT-2, blessed among the GPTs, with minor differences: layernorm -> rmsnorm, no biases, GeLU -> ReLU
  def linear(x, w):
      return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]

  def softmax(logits):
      max_val = max(val.data for val in logits)
      exps = [(val - max_val).exp() for val in logits]
      total = sum(exps)
      return [e / total for e in exps]

  def rmsnorm(x):
      ms = sum(xi * xi for xi in x) / len(x)
      scale = (ms + 1e-5) ** -0.5
      return [xi * scale for xi in x]

  def gpt(token_id, pos_id, keys, values):
      tok_emb = state_dict['wte'][token_id] # token embedding
      pos_emb = state_dict['wpe'][pos_id] # position embedding
      x = [t + p for t, p in zip(tok_emb, pos_emb)] # joint token and position embedding
      x = rmsnorm(x) # note: not redundant due to backward pass via the residual connection

      for li in range(n_layer):
          # 1) Multi-head Attention block
          x_residual = x
          x = rmsnorm(x)
          q = linear(x, state_dict[f'layer{li}.attn_wq'])
          k = linear(x, state_dict[f'layer{li}.attn_wk'])
          v = linear(x, state_dict[f'layer{li}.attn_wv'])
          keys[li].append(k)
          values[li].append(v)
          x_attn = []
          for h in range(n_head):
              hs = h * head_dim
              q_h = q[hs:hs+head_dim]
              k_h = [ki[hs:hs+head_dim] for ki in keys[li]]
              v_h = [vi[hs:hs+head_dim] for vi in values[li]]
              attn_logits = [sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim**0.5 for t in range(len(k_h))]
              attn_weights = softmax(attn_logits)
              head_out = [sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h))) for j in range(head_dim)]
              x_attn.extend(head_out)
          x = linear(x_attn, state_dict[f'layer{li}.attn_wo'])
          x = [a + b for a, b in zip(x, x_residual)]
          # 2) MLP block
          x_residual = x
          x = rmsnorm(x)
          x = linear(x, state_dict[f'layer{li}.mlp_fc1'])
          x = [xi.relu() for xi in x]
          x = linear(x, state_dict[f'layer{li}.mlp_fc2'])
          x = [a + b for a, b in zip(x, x_residual)]

      logits = linear(x, state_dict['lm_head'])
      return logits

  # Let there be Adam, the blessed optimizer and its buffers
  learning_rate, beta1, beta2, eps_adam = 0.01, 0.85, 0.99, 1e-8
  m = [0.0] * len(params) # first moment buffer
  v = [0.0] * len(params) # second moment buffer

  for step in range(num_steps):

      # Take single document, tokenize it, surround it with BOS special token on both sides
      doc = docs[step % len(docs)]
      tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
      n = min(block_size, len(tokens) - 1)

      # Forward the token sequence through the model, building up the computation graph all the way to the loss
      keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
      losses = []
      for pos_id in range(n):
          token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
          logits = gpt(token_id, pos_id, keys, values)
          probs = softmax(logits)
          loss_t = -probs[target_id].log()
          losses.append(loss_t)
      loss = (1 / n) * sum(losses) # final average loss over the document sequence. May yours be low.

      # Backward the loss, calculating the gradients with respect to all model parameters
      loss.backward()

      # Adam optimizer update: update the model parameters based on the corresponding gradients
      lr_t = learning_rate * (1 - step / num_steps) # linear learning rate decay
      for i, p in enumerate(params):
          m[i] = beta1 * m[i] + (1 - beta1) * p.grad
          v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2
          m_hat = m[i] / (1 - beta1 ** (step + 1))
          v_hat = v[i] / (1 - beta2 ** (step + 1))
          p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
          p.grad = 0

      print(f"step {step+1:4d} / {num_steps:4d} | loss {loss.data:.4f}", end='\r')

  # Inference: may the model babble back to us
  print("\n--- inference (new, hallucinated names) ---")
  for sample_idx in range(20):
      keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
      token_id = BOS
      sample = []
      for pos_id in range(block_size):
          logits = gpt(token_id, pos_id, keys, values)
          probs = softmax([l / temperature for l in logits])
          token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
          if token_id == BOS:
              break
          sample.append(uchars[token_id])
      print(f"sample {sample_idx+1:2d}: {''.join(sample)}")
print(code_karpathy())
print(f"Temps d'exécution: {time.time() - t0:.2f}s")
return None

num docs: 32033
vocab size: 27
num params: 4192


#**Partie 2 : tests en modifiant certains paramètres**

##Modification de la taille du vecteur : n_embd

Taille de base (choisie par Karpathy) : 16

Tests avec 4, 8 et 32

In [ ]:
print(f'Les prénoms obtenus avec un vecteur de taille 4 sont: ')
print(f'{code_karpathy(1,4,1000,0.5)}')


In [ ]:
print(f'Les prénoms obtenus avec un vecteur de taille 8 sont: ')
print(f'{code_karpathy(1,8,1000,0.5)}')

In [ ]:
print(f'Les prénoms obtenus avec un vecteur de taille 32 sont: ')
print(f'{code_karpathy(1,32,1000,0.5)}')


##Augmentation du nombre de répétitions du transformer:

Nombre d'itération de base : 1

Test avec : 2

In [ ]:
print(f'Les prénoms obtenus lorsque l'+'on'+'répète 2 fois le transformer sont')
print(f'{code_karpathy(2,16,1000,0.5)}')

##Modification du nombre de bouble d'entraînement

Nombre de boucle de base : 1000

Tests avec : 100, 500, 2000

In [ ]:
print(f'Les prénoms obtenus lorsqu'+'on répète 100 fois la boucle d'+'entraînement sont :')
print(f'{code_karpathy(1,16,100,0.5)}')

In [ ]:
print(f'Les prénoms obtenus lorsqu'+'on répète 500 fois la boucle d'+'entraînement sont :')
print(f'{code_karpathy(1,16,500,0.5)}')

In [ ]:
print(f'Les prénoms obtenus lorsqu'+'on répète 2000 fois la boucle d'+'entraînement sont :')
print(f'{code_karpathy(1,16,2000,0.5)}')

##Modification de la temperature :

Temperature de base: 0.5

Tests avec: 0.05, 0.2, 1

In [ ]:
print(f'Les prénoms obtenus lorsque la temperature est de 0.05 sont :')
print(f'{code_karpathy(1,16,1000,0.05)}')

In [ ]:
print(f'Les prénoms obtenus lorsque la temperature est de 0.2 sont :')
print(f'{code_karpathy(1,16,1000,0.2)}')

In [ ]:
print(f'Les prénoms obtenus lorsque la temperature est de 1 sont :')
print(f'{code_karpathy(1,16,1000,1)}')